In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp03/ex_7.py ──
"""
Shared infrastructure for MLFP03 Exercise 7 — Kailash Workflows, DataFlow
Persistence, Hyperparameter Search, and Model Registry.

Contains: leak-free dataset loading, fixed dev/test frames, FeatureSchema
construction, registry setup and artefact loading, metric computation, the
production quality gate, illustrative ROI helpers, DB URL resolution and
pipeline-audit utilities. Technique-specific code (workflow node wiring, search space
definitions, registry lifecycle transitions) lives in the per-technique files.

Available after ``uv sync`` from any directory.
"""

import os
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    log_loss,
    roc_auc_score,
)

from kailash_ml.types import FeatureField, FeatureSchema


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
load_dotenv()

RANDOM_SEED: int = 42
TARGET_COLUMN: str = "default"
DATASET_NAME: str = "sg_credit_scoring"
DATASET_FILE: str = "sg_credit_scoring.parquet"

# Columns that MUST NOT be model inputs (Lesson 3.1 leakage rule):
#   customer_id              — a row identifier, not a property of the applicant
#   future_default_indicator — recorded AFTER the loan outcome is known; it
#                              agrees with ``default`` on ~99% of rows, so a
#                              model that sees it "predicts" default by
#                              reading the answer (Exercise 4 screens for it).
CREDIT_NON_FEATURE_COLUMNS: tuple[str, ...] = ("customer_id", "future_default_indicator")

# Output directory for artefacts (audit trails, evaluation tables)
OUTPUT_DIR = Path("outputs") / "mlfp03_ex7"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# DataFlow persistence URL. SQLite is hermetic — every student gets a fresh
# DB file per run. In production we would read this from the environment.
#
# NOTE: The modern dataflow sqlite adapter interprets `sqlite:///relative`
# as an absolute `/relative` path (breaking old sqlite URL conventions).
# We therefore resolve to an absolute path and use the `sqlite:////abs`
# four-slash form so the behaviour is identical on every working dir.
_DB_ABS_PATH = (OUTPUT_DIR / "mlfp03_models.db").resolve()
DB_URL: str = os.environ.get(
    "MLFP03_EX7_DB_URL", f"sqlite:///{_DB_ABS_PATH.as_posix()}"
)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit scoring (from MLFP02)
# ════════════════════════════════════════════════════════════════════════


def load_credit_frame() -> pl.DataFrame:
    """Load the Singapore credit scoring dataset as a polars DataFrame.

    Columns: demographic + bureau features, with ``default`` (0/1) target.
    ``CREDIT_NON_FEATURE_COLUMNS`` (row ID + post-outcome leak) are dropped
    here so every downstream split, schema and pipeline inherits the fix.
    """
    loader = MLFPDataLoader()
    return loader.load("mlfp02", DATASET_FILE).drop(CREDIT_NON_FEATURE_COLUMNS)


def prepare_credit_frames(
    credit: pl.DataFrame | None = None, *, seed: int = RANDOM_SEED
) -> tuple[pl.DataFrame, pl.DataFrame, list[str]]:
    """Return ``(dev_frame, test_frame, feature_columns)`` for honest model selection.

    ``dev_frame`` (80%) is everything model SELECTION may touch —
    hyperparameter search, grid baselines, early stopping; TrainingPipeline
    carves its own validation holdout out of it. ``test_frame`` (20%) is
    touched exactly once, to report the chosen model. The 20% is held out
    BEFORE the preprocessing is fitted, so its rows shape no imputation or
    encoding statistic. Both carry a unique
    ``application_id`` so they satisfy the same ``FeatureSchema``.
    """
    if credit is None:
        credit = load_credit_frame()

    # Split FIRST, then fit imputation/encoding on the training rows only:
    # PreprocessingPipeline.setup() on the whole frame would fit them on the
    # test rows too (it splits only after fitting).
    result = split_then_preprocess(
        credit,
        target=TARGET_COLUMN,
        test_size=0.2,
        seed=seed,
        normalize=False,
        categorical_encoding="ordinal",
    )
    n_dev = result.train_data.height
    dev = result.train_data.with_columns(
        pl.int_range(0, n_dev, dtype=pl.Int64).alias("application_id")
    )
    test = result.test_data.with_columns(
        pl.int_range(n_dev, n_dev + result.test_data.height, dtype=pl.Int64).alias(
            "application_id"
        )
    )
    feature_columns = [
        c for c in dev.columns if c not in (TARGET_COLUMN, "application_id")
    ]
    return dev, test, feature_columns


def credit_feature_schema(feature_columns: list[str]) -> FeatureSchema:
    """Build a FeatureSchema matching ``prepare_credit_frames`` output."""
    return FeatureSchema(
        name="credit_model_input",
        features=[FeatureField(name=f, dtype="float64") for f in feature_columns],
        entity_id_column="application_id",
    )


async def build_training_registry(db_url: str | None = None):
    """Create + initialise a kailash-ml ModelRegistry for TrainingPipeline.

    Returns ``(registry, connection_manager)``. Caller owns ``connection_manager.close()``.
    """
    from kailash.db import ConnectionManager
    from kailash_ml import ModelRegistry

    conn = ConnectionManager(db_url or DB_URL)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return registry, conn


# ════════════════════════════════════════════════════════════════════════
# METRICS
# ════════════════════════════════════════════════════════════════════════


def compute_classification_metrics(
    y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray
) -> dict[str, float]:
    """Return accuracy, f1, auc_roc, auc_pr, log_loss."""
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "auc_pr": float(average_precision_score(y_true, y_proba)),
        "log_loss": float(log_loss(y_true, y_proba)),
    }


def print_metric_block(title: str, metrics: dict[str, float]) -> None:
    print(f"\n=== {title} ===")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")


# ════════════════════════════════════════════════════════════════════════
# QUALITY GATE — what a candidate model must clear before production
# ════════════════════════════════════════════════════════════════════════
# Both thresholds are relative to what "no skill" looks like on THIS data:
#   - AUC-ROC 0.5 is random ranking; we demand a clear margin above it.
#   - AUC-PR (average precision) of a random scorer equals the default
#     rate (~13% here); we demand at least twice that.
# Leak-free LightGBM on this dataset lands around AUC 0.77 / AUC-PR 0.37,
# so a sound model passes and a broken one (e.g. shuffled labels) fails.

PROMOTION_MIN_AUC: float = 0.75
PROMOTION_MIN_AP_MULTIPLE: float = 2.0  # × default rate


def promotion_gate(metrics: dict[str, float], default_rate: float) -> dict[str, Any]:
    """Evaluate the production quality gate.

    ``metrics`` uses ``compute_classification_metrics`` names
    (``auc_roc``, ``auc_pr``) computed on rows the model never trained on.
    Returns a JSON-serialisable dict: thresholds, observed values, verdict.
    """
    min_ap = PROMOTION_MIN_AP_MULTIPLE * default_rate
    auc = float(metrics["auc_roc"])
    ap = float(metrics["auc_pr"])
    return {
        "auc_roc": auc,
        "auc_pr": ap,
        "min_auc_roc": PROMOTION_MIN_AUC,
        "min_auc_pr": min_ap,
        "promote": bool(auc >= PROMOTION_MIN_AUC and ap >= min_ap),
    }


async def load_registered_model(registry: Any, name: str, version: int) -> Any:
    """Load a model artefact that THIS course's pipeline registered.

    The registry stores the fitted estimator as pickle bytes. Unpickling
    executes code, so only ever load artefacts you trained yourself.
    """
    import pickle

    artifact = await registry.load_artifact(name, version)
    return pickle.loads(artifact)


# Metrics TrainingPipeline can compute from its holdout predictions. In
# kailash-ml 2.2.2 the engine's evaluator does not pass predicted
# probabilities to the metric registry, so "average_precision",
# "log_loss" and "brier_score_loss" are skipped there; the exercises
# compute those with ``compute_classification_metrics`` on the
# registered model's probabilities instead.
ENGINE_METRICS: list[str] = ["accuracy", "f1", "auc"]


# ════════════════════════════════════════════════════════════════════════
# APPLY-PHASE BUSINESS ASSUMPTIONS (R9B) — ILLUSTRATIVE
# ════════════════════════════════════════════════════════════════════════
# Round numbers for a HYPOTHETICAL Singapore retail bank's unsecured
# lending book. They are teaching assumptions, not figures from any real
# lender, regulator or report — replace them with your institution's own
# numbers. Only the model-quality inputs (defaults caught, metrics) are
# measured by the exercise code itself.

ILLUSTRATIVE_BANK: dict[str, Any] = {
    "applications_per_year": 200_000,
    "avg_exposure_sgd": 18_000.0,  # average principal per approved loan
    "lgd": 0.65,  # loss given default on unsecured retail
    "review_budget": 0.10,  # share of applications the credit team can review
    "evidence_hours_per_retrain": 160.0,  # manual evidence-pack assembly
    "analyst_hourly_sgd": 120.0,
    "retrains_per_year": 12,
}


def defaults_caught_at_budget(
    y_true: np.ndarray, y_proba: np.ndarray, budget: float | None = None
) -> int:
    """Defaults among the top ``budget`` share of applications by model score.

    This is how a credit team actually uses a score: it can only review a
    fixed share of applications, so a better model is one that puts more
    true defaults into that reviewed slice.
    """
    budget = ILLUSTRATIVE_BANK["review_budget"] if budget is None else budget
    k = max(1, int(round(budget * len(y_true))))
    top = np.argsort(-np.asarray(y_proba))[:k]
    return int(np.asarray(y_true)[top].sum())


def annual_loss_avoided(extra_defaults_caught: float, n_scored: int) -> float:
    """Scale extra defaults caught on ``n_scored`` test rows to a year (S$)."""
    b = ILLUSTRATIVE_BANK
    per_year = extra_defaults_caught * b["applications_per_year"] / n_scored
    return float(per_year * b["avg_exposure_sgd"] * b["lgd"])


def evidence_prep_savings() -> float:
    """Annual analyst cost of hand-assembling model evidence packs (S$)."""
    b = ILLUSTRATIVE_BANK
    return float(
        b["evidence_hours_per_retrain"] * b["analyst_hourly_sgd"] * b["retrains_per_year"]
    )


def headline_roi_text(loss_avoided_sgd: float | None = None) -> str:
    """Plain-text ROI block for the Apply phases (ILLUSTRATIVE assumptions).

    ``loss_avoided_sgd`` must come from a measured comparison (see
    03_hyperparameter_search.py); when it is not supplied the line says so
    instead of inventing a number.
    """
    b = ILLUSTRATIVE_BANK
    audit = evidence_prep_savings()
    lines = [
        "  (illustrative assumptions for a hypothetical Singapore retail bank)",
        f"  Applications scored:  {b['applications_per_year']:,} / yr",
        f"  Retrains:             {b['retrains_per_year']} / yr",
        f"  Evidence-pack prep:   ~S${audit/1e3:,.0f}k / yr of analyst time that a",
        "                        persisted, replayable pipeline removes",
    ]
    if loss_avoided_sgd is None:
        lines.append(
            "  Loss avoided:         not claimed here — measured in "
            "03_hyperparameter_search.py"
        )
    else:
        lines.append(
            f"  Loss avoided:         ~S${loss_avoided_sgd/1e6:,.2f}M / yr "
            "(measured lift × assumptions)"
        )
    return "\n".join(lines)


# ════════════════════════════════════════════════════════════════════════
# PIPELINE AUDIT HELPERS
# ════════════════════════════════════════════════════════════════════════


def audit_trail_row(
    *,
    stage: str,
    detail: str,
    run_id: str,
) -> dict[str, Any]:
    """Structured audit row used by the orchestrated pipeline."""
    return {"stage": stage, "detail": detail, "run_id": run_id}




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 7.1: Kailash WorkflowBuilder + Custom Nodes
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Write custom nodes with `@register_node()` (Node and AsyncNode)
#   - Add a PythonCodeNode for a small inline step and a SwitchNode for
#     a conditional branch
#   - Wire node outputs to node inputs with `add_connection(...)`
#   - Execute the DAG with `runtime.execute(workflow.build())` and read
#     every node's real output plus the run_id
#   - Promote a model to production only when the quality gate passes
#
# PREREQUISITES: MLFP03 Exercises 1-6, MLFP02 preprocessing
# ESTIMATED TIME: ~45 min
#
# 5-PHASE R10:
#   1. Theory     — why workflows beat hand-rolled scripts
#   2. Build      — custom nodes + load→preprocess→train→evaluate→gate
#   3. Train      — LocalRuntime.execute(workflow.build())
#   4. Visualise  — trace every node's output and the branch taken
#   5. Apply      — monthly credit-model retraining at a Singapore bank
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from typing import Any

import polars as pl

from kailash.nodes.base import Node, NodeParameter, register_node
from kailash.nodes.base_async import AsyncNode
from kailash.runtime import LocalRuntime
from kailash.workflow.builder import WorkflowBuilder
from kailash_ml import TrainingPipeline
from kailash_ml.engines.training_pipeline import EvalSpec, ModelSpec
from kailash_ml.interop import to_sklearn_input



## THEORY — Why WorkflowBuilder?


In [ ]:
# A hand-rolled training script is "just Python" until the day it isn't.
# When you come back six months later and ask "which model went into
# production on March 12?", you need four things the script cannot give
# you without re-implementation:
#
#   1. A NAME for each step ("preprocess", "train", "gate") so you can
#      reference it in an audit row.
#   2. A RUN_ID for every execution so the June run is distinguishable
#      from the March run.
#   3. DEPENDENCIES between steps expressed as edges, not as "the order
#      functions happen to appear in the file."
#   4. A SINGLE EXECUTION ENTRYPOINT: `runtime.execute(workflow.build())`.
#
# Three kinds of node build this exercise's DAG:
#   - CUSTOM nodes: a class decorated with `@register_node()` that
#     subclasses `Node` (sync `run`) or `AsyncNode` (`async_run`, for
#     engines such as TrainingPipeline that talk to a database).
#   - PythonCodeNode: a few lines of inline code; its output is the dict
#     you assign to `result`, so downstream nodes read `result`.
#   - SwitchNode: routes its `input_data` to `true_output` or
#     `false_output`; only the branch that receives data runs.
#
# One rule shapes the design: node outputs must be JSON-serialisable.
# Heavy objects therefore travel as REFERENCES — a parquet path for the
# data, a registry name + version for the model — exactly like a
# production pipeline that passes artefact pointers, not objects.



## TASK 2 — BUILD the custom nodes and the DAG


Data-loading node: leak-free credit table → parquet artefact.


The test rows are held out BEFORE the pipeline is fitted, so the
    imputation and encoding are learned from the dev rows only. The dev
    frame is for training (TrainingPipeline takes its own holdout from
    it); the test frame is touched only by the evaluation node.


TrainingPipeline holds out 20% of the dev frame, fits LightGBM on the
    rest, scores that holdout and registers the model (stage: staging).
    The node returns the metrics and the registry reference — not the
    model object.


Evaluation node: score the registered model on the untouched test frame.


Promotion node: moves the registered version staging → production.


min_ap = {PROMOTION_MIN_AP_MULTIPLE} * default_rate
auc = test_metrics['auc_roc']
ap = test_metrics['auc_pr']
result = {{'auc_roc': auc, 'auc_pr': ap,
          'min_auc_roc': {PROMOTION_MIN_AUC}, 'min_auc_pr': min_ap,
          'promote': auc >= {PROMOTION_MIN_AUC} and ap >= min_ap,
          'model_name': model_name, 'model_version': model_version}}


In [ ]:
EXPERIMENT_NAME = "credit_default_workflow"
RAW_PATH = OUTPUT_DIR / "ex7_01_raw.parquet"
DEV_PATH = OUTPUT_DIR / "ex7_01_dev.parquet"
TEST_PATH = OUTPUT_DIR / "ex7_01_test.parquet"


@register_node()
class CreditLoadNode(Node):

    def get_parameters(self) -> dict[str, NodeParameter]:
        return {}

    def run(self, **kwargs: Any) -> dict[str, Any]:
        raw = load_credit_frame()
        raw.write_parquet(RAW_PATH)
        return {
            "raw_path": str(RAW_PATH),
            "rows": raw.height,
            "default_rate": float(raw["default"].mean()),
        }


@register_node()
class CreditPreprocessNode(Node):

    def get_parameters(self) -> dict[str, NodeParameter]:
        return {
            "raw_path": NodeParameter(name="raw_path", type=str, required=True),
            "seed": NodeParameter(
                name="seed", type=int, required=False, default=RANDOM_SEED
            ),
        }

    def run(self, **kwargs: Any) -> dict[str, Any]:
        raw = pl.read_parquet(kwargs["raw_path"])
        # TODO: split the raw frame into (dev, test, feature_columns) with the
        # shared helper that runs PreprocessingPipeline, passing the seed.
        # Hint: prepare_credit_frames(<frame>, seed=...)
        dev, test, feature_columns = ____
        dev.write_parquet(DEV_PATH)
        test.write_parquet(TEST_PATH)
        return {
            "dev_path": str(DEV_PATH),
            "test_path": str(TEST_PATH),
            "feature_columns": feature_columns,
        }


@register_node()
class TrainingPipelineNode(AsyncNode):

    def get_parameters(self) -> dict[str, NodeParameter]:
        return {
            "dev_path": NodeParameter(name="dev_path", type=str, required=True),
            "feature_columns": NodeParameter(
                name="feature_columns", type=list, required=True
            ),
            "hyperparameters": NodeParameter(
                name="hyperparameters", type=dict, required=True
            ),
            "experiment_name": NodeParameter(
                name="experiment_name", type=str, required=True
            ),
        }

    async def async_run(self, **kwargs: Any) -> dict[str, Any]:
        dev = pl.read_parquet(kwargs["dev_path"])
        registry, conn = await build_training_registry()
        try:
            # TODO: build the TrainingPipeline engine (no feature store here).
            # Hint: TrainingPipeline(feature_store=..., registry=...)
            pipeline = ____
            result = await pipeline.train(
                data=dev,
                schema=credit_feature_schema(kwargs["feature_columns"]),
                model_spec=ModelSpec(
                    model_class="lightgbm.LGBMClassifier",
                    framework="lightgbm",
                    hyperparameters=kwargs["hyperparameters"],
                ),
                eval_spec=EvalSpec(
                    metrics=ENGINE_METRICS, split_strategy="holdout", test_size=0.2
                ),
                experiment_name=kwargs["experiment_name"],
            )
        finally:
            await conn.close()
        if result.model_version is None:
            raise RuntimeError("TrainingPipeline did not register the model")
        return {
            "holdout_metrics": {k: float(v) for k, v in result.metrics.items()},
            "model_name": kwargs["experiment_name"],
            "model_version": int(result.model_version.version),
        }


@register_node()
class EvaluateNode(AsyncNode):

    def get_parameters(self) -> dict[str, NodeParameter]:
        return {
            "test_path": NodeParameter(name="test_path", type=str, required=True),
            "feature_columns": NodeParameter(
                name="feature_columns", type=list, required=True
            ),
            "model_name": NodeParameter(name="model_name", type=str, required=True),
            "model_version": NodeParameter(
                name="model_version", type=int, required=True
            ),
        }

    async def async_run(self, **kwargs: Any) -> dict[str, Any]:
        test = pl.read_parquet(kwargs["test_path"])
        X_test, y_test, _ = to_sklearn_input(
            test,
            feature_columns=kwargs["feature_columns"],
            target_column=TARGET_COLUMN,
        )
        registry, conn = await build_training_registry()
        try:
            # TODO: load the model the train node registered (name + version).
            # Hint: the shared load_registered_model(registry, name, version) is async
            model = await ____
        finally:
            await conn.close()
        # TODO: probability of default (class 1) for every test row
        # Hint: predict_proba returns one column per class
        y_proba = ____
        y_pred = (y_proba >= 0.5).astype(int)
        return {"test_metrics": compute_classification_metrics(y_test, y_pred, y_proba)}


@register_node()
class PromoteNode(AsyncNode):

    def get_parameters(self) -> dict[str, NodeParameter]:
        return {"payload": NodeParameter(name="payload", type=dict, required=True)}

    async def async_run(self, **kwargs: Any) -> dict[str, Any]:
        p = kwargs["payload"]
        registry, conn = await build_training_registry()
        try:
            promoted = await registry.promote_model(
                p["model_name"],
                p["model_version"],
                # TODO: the target stage for a model that passed the gate
                # Hint: registry stages are staging / shadow / production / archived
                ____,
                reason=(
                    f"Quality gate passed on the test frame: "
                    f"AUC-ROC {p['auc_roc']:.4f} >= {p['min_auc_roc']}, "
                    f"AUC-PR {p['auc_pr']:.4f} >= {p['min_auc_pr']:.4f}"
                ),
            )
        finally:
            await conn.close()
        return {"stage": promoted.stage, "model_version": p["model_version"]}


# The quality gate is a PythonCodeNode: a few inline lines whose `result`
# dict becomes the SwitchNode's input. `test_metrics`, `default_rate`,
# `model_name` and `model_version` arrive through connections.

HYPERPARAMETERS = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 5,
    "num_leaves": 31,
    "min_child_samples": 40,
    "random_state": RANDOM_SEED,
    "verbose": -1,
}

workflow = WorkflowBuilder()
workflow.add_node("CreditLoadNode", "load", {})
workflow.add_node("CreditPreprocessNode", "preprocess", {"seed": RANDOM_SEED})
workflow.add_node(
    "TrainingPipelineNode",
    "train",
    {"hyperparameters": HYPERPARAMETERS, "experiment_name": EXPERIMENT_NAME},
)
workflow.add_node("EvaluateNode", "evaluate", {})
workflow.add_node("PythonCodeNode", "gate_check", {"code": GATE_CODE})
# TODO: configure the SwitchNode to route on the gate_check verdict.
# Hint: {"condition_field": <key in the input dict>, "operator": "==", "value": ...}
workflow.add_node("SwitchNode", "gate", ____)
workflow.add_node("PromoteNode", "promote", {})
workflow.add_node(
    "PythonCodeNode",
    "hold",
    {"code": "result = dict(payload, stage='staging', held=True)"},
)

# Edges: add_connection(from_node, from_output, to_node, to_input)
workflow.add_connection("load", "raw_path", "preprocess", "raw_path")
workflow.add_connection("preprocess", "dev_path", "train", "dev_path")
workflow.add_connection("preprocess", "feature_columns", "train", "feature_columns")
workflow.add_connection("preprocess", "test_path", "evaluate", "test_path")
workflow.add_connection("preprocess", "feature_columns", "evaluate", "feature_columns")
workflow.add_connection("train", "model_name", "evaluate", "model_name")
workflow.add_connection("train", "model_version", "evaluate", "model_version")
workflow.add_connection("evaluate", "test_metrics", "gate_check", "test_metrics")
workflow.add_connection("load", "default_rate", "gate_check", "default_rate")
workflow.add_connection("train", "model_name", "gate_check", "model_name")
workflow.add_connection("train", "model_version", "gate_check", "model_version")
# TODO: feed the PythonCodeNode's output dict into the SwitchNode.
# Hint: PythonCodeNode outputs live under "result"; SwitchNode reads "input_data"
workflow.add_connection(____)
# TODO: which SwitchNode output should reach the promote node?
# Hint: SwitchNode emits "true_output" and "false_output"
workflow.add_connection("gate", ____, "promote", "payload")
workflow.add_connection("gate", "false_output", "hold", "payload")



## TASK 3 — TRAIN via runtime.execute(workflow.build())


In [ ]:
# LocalRuntime validates the DAG, runs each node in dependency order and
# returns (results, run_id). `results[node_id]` is that node's output
# dict; a node on the branch the SwitchNode did NOT take has no entry.
# If any node fails, execute() raises — there is no silent fallback.

print("\n" + "=" * 70)
print("  Executing the credit-scoring workflow")
print("=" * 70)

# conditional_execution="skip_branches" makes the runtime SKIP every node
# on the SwitchNode branch that received no data (the default
# "route_data" mode would still try to feed it None). In kailash 2.44.1
# this mode also logs one "Error tracking conditional execution
# performance" line from its internal metrics hook; execution itself is
# unaffected. The context manager releases the runtime's resources.
with LocalRuntime(conditional_execution="skip_branches") as runtime:
    # TODO: execute the BUILT workflow
    # Hint: runtime.execute(...) takes workflow.build(), not the builder
    results, run_id = runtime.execute(____)
print(f"  run_id: {run_id}")

train_out = results["train"]
test_metrics = results["evaluate"]["test_metrics"]
gate_in = results["gate_check"]["result"]
branch_taken = "promote" if results.get("promote") is not None else "hold"



### Checkpoint


In [ ]:
assert run_id is not None, "Task 3: runtime.execute must return a run_id"
assert results["load"]["rows"] > 0, "Task 3: the load node must read rows"
assert train_out["model_version"] >= 1, "Task 3: the model must be registered"
assert 0.5 < test_metrics["auc_roc"] <= 1.0, "Task 3: test AUC must beat random"
assert (results.get("promote") is None) != (
    results.get("hold") is None
), "Task 3: exactly one SwitchNode branch must run"
assert (branch_taken == "promote") == gate_in["promote"], (
    "Task 3: the branch taken must match the gate verdict"
)
print("\n[ok] Checkpoint passed — custom nodes, PythonCodeNode and SwitchNode ran\n")



## TASK 4 — VISUALISE the executed DAG


In [ ]:
print("Executed DAG (node -> output):")
print(
    f"  load        -> {results['load']['rows']:,} rows, "
    f"default rate {results['load']['default_rate']:.2%}"
)
print(
    f"  preprocess  -> {len(results['preprocess']['feature_columns'])} features; "
    f"dev + test frames written to {OUTPUT_DIR}"
)
print(
    f"  train       -> {train_out['model_name']} v{train_out['model_version']} "
    "(registered: staging)"
)
print(
    f"  evaluate    -> test AUC-ROC {test_metrics['auc_roc']:.4f}, "
    f"AUC-PR {test_metrics['auc_pr']:.4f}"
)
print(
    f"  gate_check  -> needs AUC-ROC >= {gate_in['min_auc_roc']} and "
    f"AUC-PR >= {gate_in['min_auc_pr']:.4f}: promote={gate_in['promote']}"
)
print(f"  gate        -> condition_result={results['gate']['condition_result']}")
if branch_taken == "promote":
    print(f"  promote     -> stage={results['promote']['stage']}")
    print("  hold        -> skipped (branch not taken)")
else:
    print("  promote     -> skipped (branch not taken)")
    print(f"  hold        -> stays in {results['hold']['result']['stage']}")

print_metric_block(
    "TrainingPipeline holdout metrics (train node, from the dev frame)",
    train_out["holdout_metrics"],
)
print_metric_block("Untouched test-frame metrics (evaluate node)", test_metrics)



## TASK 5 — APPLY: Monthly Credit-Model Retraining at a Singapore Bank


In [ ]:
# SCENARIO (illustrative): a Singapore retail bank's credit-risk team
# retrains its default model on the first Monday of every month. Today
# the retraining is a notebook owned by one analyst; the bank's model-risk
# policy asks for every production model to be reproducible and for each
# promotion to be justified — and a notebook cannot show either.
#
# Converting the notebook to this workflow gives:
#   - A single named entrypoint the retraining scheduler can invoke
#   - A run_id per execution, which maps 1:1 to an audit row
#   - A promotion that happens ONLY when the gate node says so, with the
#     gate's numbers written into the registry's promotion reason
#   - No hidden state — every node's input is a config value or an edge
print("\n" + "=" * 70)
print("  APPLY: Monthly Retraining as a Workflow")
print("=" * 70)
print(headline_roi_text())



## REFLECTION


[x] Wrote custom nodes with @register_node() — Node and AsyncNode
  [x] Used a PythonCodeNode for the gate and a SwitchNode to branch
  [x] Wired outputs to inputs with add_connection(...)
  [x] Executed with runtime.execute(workflow.build()) — run_id {run_id[:8]}
  [x] Branch taken this run: {branch_taken}

  KEY INSIGHT: The workflow is documentation you can execute. Every edge
  is machine-readable, every node's output is inspectable, and the
  promotion decision is a node — not a line someone forgot to run.

  Next: 02_dataflow_persistence.py — stop returning metrics as a dict
  and start writing them to a DataFlow-managed database table.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

